# 01 · Ingredient pairings by region and season

Question: **which ingredients go together on fine-dining menus, and how does that shift by region and hemisphere-relative season?**

Data source, set with `MENUDB_ANALYSIS_SOURCE`:
- `db`: Postgres (locally, or wherever `DATABASE_URL` is reachable);
- `bucket`: the snapshots bucket's `extracted/` and `imported/` objects, no database (works in cloud sessions; dates unknown, so no seasons);
- `synthetic` (default): `tests/fixtures/synthetic_menus.json`, invented data with planted patterns.

Only derived data appears here: ingredient names, counts and scores. **Clear outputs before committing** when running on real data; the repo is public and results stay private until the stance in `docs/PLAN.md` is reviewed.

Run: `uv sync --group analysis`, then `uv run jupyter nbconvert --to notebook --execute notebooks/01_pairings.ipynb --output /tmp/out.ipynb`, or open it in Jupyter.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from menudb import analysis as a

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
SOURCE = os.environ.get("MENUDB_ANALYSIS_SOURCE", "synthetic")

if SOURCE == "db":
    from menudb.db import get_session
    with get_session() as session:
        rows, restaurants = a.load_db(session)
elif SOURCE == "bucket":
    rows, restaurants = a.load_bucket(ROOT / "data/seed/restaurants.csv")
else:
    rows, restaurants = a.load_synthetic(ROOT / "tests/fixtures/synthetic_menus.json")

df = pd.DataFrame([r.__dict__ | {"season": r.season, "hemisphere": r.hemisphere} for r in rows])
print(f"source={SOURCE}: {df.restaurant.nunique()} restaurants, {df.menu.nunique()} menus, "
      f"{df.groupby(['restaurant', 'menu', 'course']).ngroups} courses, {df.ingredient.nunique()} ingredients")

## Coverage

Shown next to every result: the seed is a curated list, and many top restaurants publish no menu, so any pattern below describes *restaurants that publish menus*.

In [ ]:
coverage = pd.DataFrame(a.coverage(restaurants, rows)).set_index("region")
coverage

## Most common ingredients

`share`: restaurant-weighted fraction of menus using the ingredient (each restaurant weighs 1, split over its distinct menus).

In [ ]:
freq = pd.DataFrame(a.ingredient_frequency(rows))
ax = freq.head(20).iloc[::-1].plot.barh(x="ingredient", y="share", legend=False, figsize=(6, 6))
ax.set_xlabel("share of restaurants' menus")
plt.tight_layout()
freq.head(20)

## By region

In [ ]:
by_region = pd.DataFrame(a.ingredient_frequency(rows, by="region"))
by_region.groupby("group").head(5).set_index(["group", "ingredient"])

## By hemisphere-relative season

Southern-Hemisphere dates shift six months (`seasons.py`); restaurants inside the tropics are grouped as `tropical`.

In [ ]:
by_season = pd.DataFrame(a.ingredient_frequency(rows, by="season"))
pivot = by_season.pivot_table(index="ingredient", columns="group", values="share", fill_value=0)
pivot.loc[pivot.max(axis=1).sort_values(ascending=False).index[:15]].round(2)

## Pairings

Course-level co-occurrence. `lift` > 1 means the pair shares a course more often than chance; `pmi` = log2(lift). Pairs need at least `MIN_RESTAURANTS` restaurants, so one chef's signature dish doesn't count as a pattern.

In [ ]:
MIN_RESTAURANTS = 2
pairs = pd.DataFrame(a.pair_stats(rows, min_restaurants=MIN_RESTAURANTS))
pairs.head(20).round(3)

In [ ]:
top = pairs.head(15).iloc[::-1]
ax = top.plot.barh(x=None, y="pmi", legend=False, figsize=(6, 5))
ax.set_yticklabels([f"{x.a} + {x.b}" for x in top.itertuples()])
ax.set_xlabel("PMI (bits)")
plt.tight_layout()

## Pairings within each region

The same statistic computed per region; small regions will be noisy, so read alongside the coverage table.

In [ ]:
region_pairs = []
for region, group in df.groupby("region"):
    sub = [r for r in rows if r.region == region]
    for p in a.pair_stats(sub, min_restaurants=min(MIN_RESTAURANTS, group.restaurant.nunique()))[:5]:
        region_pairs.append({"region": region} | p)
pd.DataFrame(region_pairs).round(3)